In [1]:
# import libraries for data cleaning
import numpy as np
import pandas as pd

In [10]:
matches = pd.read_csv('ipl_matches.csv')
balls = pd.read_csv('IPL_ball_by_ball.csv')

print(matches.shape)
print(balls.shape)

(1169, 19)
(243815, 22)


In [91]:
df_full = balls.copy()

In [94]:
df_full = pd.read_csv('IPL_ball_by_ball.csv')

In [11]:
matches.columns


Index(['team1', 'team2', 'match_date', 'toss_winner', 'toss_decision',
       'winner', 'player_of_match', 'venue', 'city', 'team1_players',
       'team2_players', 'season', 'match_number', 'match_type', 'result',
       'result_margin', 'target_runs', 'target_overs', 'super_over'],
      dtype='object')

In [9]:
balls.columns

Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type',
       'other_player_dismissed'],
      dtype='object')

In [49]:
def normalize_teams(df):
    team_map = {
        'Delhi Daredevils': 'Delhi Capitals',
        'Rising Pune Supergiants': 'Rising Pune Supergiant'
    }
    for col in ['batting_team', 'bowling_team', 'winner']:
        if col in df.columns:
            df[col] = df[col].replace(team_map)
    return df

In [50]:
df = normalize_teams(df)

In [51]:
print(df['batting_team'].isin(df['winner']).value_counts())


batting_team
False    117531
Name: count, dtype: int64


In [12]:
def normalize_team_names(df, col):
    df[col] = df[col].replace({
        'Delhi Daredevils': 'Delhi Capitals',
        'Rising Pune Supergiants': 'Rising Pune Supergiant'
    })
    return df

for col in ['team1', 'team2', 'winner']:
    matches = normalize_team_names(matches, col)

for col in ['batting_team', 'bowling_team']:
    balls = normalize_team_names(balls, col)


In [15]:
matches = matches.reset_index(drop=True)
matches['match_id'] = matches.index + 1




In [16]:
matches_subset = matches[['match_id', 'city', 'winner']]

df = balls.merge(
    matches_subset,
    on='match_id',
    how='left'
)

print("Merged data shape:", df.shape)
df[['match_id', 'batting_team', 'bowling_team', 'city', 'winner']].head()


Merged data shape: (243815, 24)


,match_id,batting_team,bowling_team,city,winner
0,335982,Kolkata Knight Riders,Royal Challengers Bangalore,NaN,NaN
1,335982,Kolkata Knight Riders,Royal Challengers Bangalore,NaN,NaN
2,335982,Kolkata Knight Riders,Royal Challengers Bangalore,NaN,NaN
3,335982,Kolkata Knight Riders,Royal Challengers Bangalore,NaN,NaN
4,335982,Kolkata Knight Riders,Royal Challengers Bangalore,NaN,NaN


In [17]:
df[['city', 'winner']].isnull().sum()


,0
city,243815
winner,243815


In [95]:
df_full['total_runs'] = df_full['runs_off_bat'] + df_full['extras'].fillna(0)

df_full = normalize_teams(df_full)

In [31]:
# Batsman stats
batsman_stats = df.groupby('striker').agg(
    batsman_runs=('runs_off_bat', 'sum'),
    balls_faced=('ball', 'count')
).reset_index()

batsman_stats['batsman_strike_rate'] = (
    batsman_stats['batsman_runs'] / batsman_stats['balls_faced']
) * 100


In [32]:
# Bowler stats
bowler_stats = df.groupby('bowler').agg(
    runs_conceded=('total_runs', 'sum'),
    balls_bowled=('ball', 'count')
).reset_index()

bowler_stats['overs'] = bowler_stats['balls_bowled'] / 6
bowler_stats['bowler_economy'] = (
    bowler_stats['runs_conceded'] / bowler_stats['overs']
)


In [33]:
df = df.merge(
    batsman_stats[['striker', 'batsman_strike_rate']],
    on='striker',
    how='left'
)

df = df.merge(
    bowler_stats[['bowler', 'bowler_economy']],
    on='bowler',
    how='left'
)


In [35]:
df['batsman_strike_rate'] = df['batsman_strike_rate'].fillna(
    df['batsman_strike_rate'].mean()
)

df['bowler_economy'] = df['bowler_economy'].fillna(
    df['bowler_economy'].mean()
)



In [36]:
df['current_score'] = (
    df.groupby(['match_id', 'innings'])['total_runs']
      .cumsum()
)

In [37]:
df['is_wicket'] = df['player_dismissed'].notnull().astype(int)

df['wickets_out'] = (
    df.groupby(['match_id', 'innings'])['is_wicket']
      .cumsum()
)

In [96]:
first_innings = (
    df_full[df_full['innings'] == 1]
    .groupby('match_id')['total_runs']
    .sum()
    .reset_index()
)

first_innings['target'] = first_innings['total_runs'] + 1
first_innings.drop('total_runs', axis=1, inplace=True)

In [98]:
df_full['target'].isnull().sum()


np.int64(0)

In [99]:
df = df_full[df_full['innings'] == 2].copy()

In [100]:
df['current_score'] = df.groupby('match_id')['total_runs'].cumsum()

df['over'] = df['ball'].astype(int)
df['ball_in_over'] = ((df['ball'] - df['over']) * 10).astype(int)
df['balls_completed'] = df['over'] * 6 + df['ball_in_over']
df['balls_left'] = 120 - df['balls_completed']

df['runs_left'] = df['target'] - df['current_score']


In [101]:
df['runs_left'].isnull().sum()


np.int64(0)

In [102]:
df['required_run_rate'] = (df['runs_left'] * 6) / df['balls_left']
df.replace([np.inf, -np.inf], 0, inplace=True)


In [103]:
df['runs_left'].isnull().sum()

np.int64(0)

In [104]:
# Wickets
df['is_wicket'] = df['player_dismissed'].notnull().astype(int)
df['wickets_out'] = df.groupby('match_id')['is_wicket'].cumsum()
df['wickets_left'] = 10 - df['wickets_out']

# Run rates
df['run_rate'] = (df['current_score'] * 6) / df['balls_completed']
df['required_run_rate'] = (df['runs_left'] * 6) / df['balls_left']
df.replace([np.inf, -np.inf], 0, inplace=True)


In [105]:
win_status = (
    df.groupby('match_id')['runs_left']
      .min()
      .reset_index()
)

win_status['batting_team_won'] = (win_status['runs_left'] <= 0).astype(int)
win_status.drop('runs_left', axis=1, inplace=True)

df = df.merge(win_status, on='match_id', how='left')

print(df['batting_team_won'].value_counts())


batting_team_won
1    59992
0    57539
Name: count, dtype: int64


In [106]:
# Batsman
batsman_stats = df.groupby('striker').agg(
    batsman_runs=('runs_off_bat', 'sum'),
    balls_faced=('ball', 'count')
).reset_index()
batsman_stats['batsman_strike_rate'] = (batsman_stats['batsman_runs'] / batsman_stats['balls_faced']) * 100

# Bowler
bowler_stats = df.groupby('bowler').agg(
    runs_conceded=('total_runs', 'sum'),
    balls_bowled=('ball', 'count')
).reset_index()
bowler_stats['overs'] = bowler_stats['balls_bowled'] / 6
bowler_stats['bowler_economy'] = bowler_stats['runs_conceded'] / bowler_stats['overs']

df = df.merge(batsman_stats[['striker','batsman_strike_rate']], on='striker', how='left')
df = df.merge(bowler_stats[['bowler','bowler_economy']], on='bowler', how='left')

df['batsman_strike_rate'] = df['batsman_strike_rate'].fillna(df['batsman_strike_rate'].mean())
df['bowler_economy'] = df['bowler_economy'].fillna(df['bowler_economy'].mean())


In [108]:
final_df = df[
    [
        'batting_team',
        'bowling_team',
        'current_score',
        'runs_left',
        'balls_left',
        'wickets_left',
        'run_rate',
        'required_run_rate',
        'batsman_strike_rate',
        'bowler_economy',
        'batting_team_won'
    ]
]

In [109]:
categorical = ['batting_team', 'bowling_team']
numeric = [c for c in X.columns if c not in categorical]

In [97]:
df_full = df_full.merge(first_innings, on='match_id', how='left')

In [39]:
df = df.merge(first_innings_score, on='match_id', how='left')

In [53]:
df = df[df['innings'] == 2]

In [54]:
final_ball = (
    df.sort_values(['match_id', 'ball'])
      .groupby('match_id')
      .tail(1)
)

final_ball['batting_team_won'] = (
    final_ball['current_score'] >= final_ball['target']
).astype(int)


In [55]:
df = df.merge(
    final_ball[['match_id', 'batting_team_won']],
    on='match_id',
    how='left'
)


In [57]:
print(df.columns)

Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type',
       'other_player_dismissed', 'city', 'winner', 'total_runs',
       'current_score', 'is_wicket', 'wickets_out', 'target_x', 'over',
       'ball_in_over', 'balls_completed', 'balls_left', 'runs_left',
       'wickets_left', 'run_rate', 'required_run_rate', 'batting_team_won_x',
       'batsman_strike_rate', 'bowler_economy', 'target_y', 'target',
       'batting_team_won_y'],
      dtype='object')


In [58]:
# Drop duplicate / unwanted columns
df.drop(
    columns=[
        'target_x', 'target_y',
        'batting_team_won_x'
    ],
    inplace=True,
    errors='ignore'
)

# Rename correct win column
df.rename(
    columns={'batting_team_won_y': 'batting_team_won'},
    inplace=True
)


In [59]:
print(df.columns)
print(df['batting_team_won'].value_counts())


Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type',
       'other_player_dismissed', 'city', 'winner', 'total_runs',
       'current_score', 'is_wicket', 'wickets_out', 'over', 'ball_in_over',
       'balls_completed', 'balls_left', 'runs_left', 'wickets_left',
       'run_rate', 'required_run_rate', 'batsman_strike_rate',
       'bowler_economy', 'target', 'batting_team_won'],
      dtype='object')
batting_team_won
0    117531
Name: count, dtype: int64


In [60]:
# A team wins if runs_left ever becomes <= 0 in 2nd innings
win_status = (
    df.groupby('match_id')['runs_left']
      .min()
      .reset_index()
)

win_status['batting_team_won'] = (win_status['runs_left'] <= 0).astype(int)
win_status.drop('runs_left', axis=1, inplace=True)


In [61]:
df = df.drop(columns=['batting_team_won'], errors='ignore')

df = df.merge(
    win_status,
    on='match_id',
    how='left'
)


In [62]:
print(df['batting_team_won'].value_counts())


batting_team_won
1    59992
0    57539
Name: count, dtype: int64


In [68]:
df['over'] = df['ball'].astype(int)
df['ball_in_over'] = ((df['ball'] - df['over']) * 10).astype(int)

df['balls_completed'] = df['over'] * 6 + df['ball_in_over']
df['balls_left'] = 120 - df['balls_completed']


In [87]:
df['runs_left'] = df['target'] - df['current_score']


In [88]:
df['runs_left'].isnull().sum()

np.int64(117531)

In [89]:
df['required_run_rate'] = (df['runs_left'] * 6) / df['balls_left']
df.replace([np.inf, -np.inf], 0, inplace=True)


In [90]:
df['required_run_rate'].isnull().sum()


np.int64(117531)

In [70]:
df['wickets_left'] = 10 - df['wickets_out']


In [71]:
df['run_rate'] = (df['current_score'] * 6) / df['balls_completed']
df['required_run_rate'] = (df['runs_left'] * 6) / df['balls_left']

df.replace([np.inf, -np.inf], 0, inplace=True)


In [72]:
df['batting_team_won'] = (df['batting_team'] == df['winner']).astype(int)


In [73]:
final_df = df[
    [
        'batting_team',
        'bowling_team',
        'city',
        'current_score',
        'runs_left',
        'balls_left',
        'wickets_left',
        'run_rate',
        'required_run_rate',
        'batsman_strike_rate',
        'bowler_economy',
        'batting_team_won'
    ]
]

final_df.dropna(inplace=True)
print(final_df.shape)


(0, 12)


/tmp/ipython-input-2638575210.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.dropna(inplace=True)


## Model

In [110]:
X = final_df.drop('batting_team_won', axis=1)
y = final_df['batting_team_won']

In [111]:

print(X.shape)
print(y.value_counts())

(117531, 10)
batting_team_won
1    59992
0    57539
Name: count, dtype: int64


In [115]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

categorical = ['batting_team', 'bowling_team']
numeric = [c for c in X.columns if c not in categorical]

numeric_transformer = SimpleImputer(strategy='mean')

categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical),
        ('num', numeric_transformer, numeric)
    ]
)


In [116]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [117]:
from sklearn.linear_model import LogisticRegression

Xtr = preprocessor.fit_transform(X_train)
Xte = preprocessor.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(Xtr, y_train)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [118]:
from sklearn.metrics import accuracy_score, roc_auc_score

y_pred = model.predict(Xte)
y_prob = model.predict_proba(Xte)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


Accuracy: 0.7958055047432679
ROC-AUC: 0.8805835343189734


In [ ]:
import pickle

# Save model
with open("ipl_win_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save preprocessor
with open("ipl_preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

In [ ]:
import sklearn
print(sklearn.__version__)

1.6.1
